In [1]:
import os
import json
import requests
import cv2
from PIL import Image
from sklearn.cluster import KMeans
import numpy as np

# Groq API Configurations (replace with actual endpoint and key)
GROQ_API_URL = "https://api.groq.com/v1/infer"  # Update with the correct Groq endpoint
GROQ_API_KEY = "your_api_key_here"


def detect_texture_firmness(image_path):
    """
    Detects texture and firmness using Groq API.
    Args:
        image_path (str): Path to the input image
    Returns:
        dict: Detected texture and firmness with bounding boxes
    """
    headers = {
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json"
    }

    # Prepare image data
    with open(image_path, "rb") as f:
        image_bytes = f.read()

    # Groq API payload
    payload = {
        "task": "detect_texture_firmness",
        "image": image_bytes.decode("latin1"),
        "parameters": {
            "text_prompt": "Detect and provide bounding boxes for texture and firmness of the fruit or vegetable in this image."
        }
    }

    response = requests.post(GROQ_API_URL, headers=headers, json=payload)
    if response.status_code == 200:
        return response.json()  # Expected to return data as JSON
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return []


def detect_color(image_path):
    """
    Detects the dominant color of the fruit/vegetable in the image.
    Args:
        image_path (str): Path to the input image
    Returns:
        List[str]: List of dominant colors
    """
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = image.reshape((-1, 3))

    # Use KMeans clustering to find dominant colors
    kmeans = KMeans(n_clusters=3, random_state=42).fit(image)
    colors = kmeans.cluster_centers_.astype(int)

    # Convert RGB to color names (you can use libraries like webcolors for names)
    dominant_colors = [f"RGB({r},{g},{b})" for r, g, b in colors]
    return dominant_colors


def draw_bounding_boxes(image_path, label_data, output_folder):
    """
    Draw bounding boxes on the image with labels for texture, firmness, and color.
    Args:
        image_path (str): Path to the input image
        label_data (dict): Detected labels and bounding boxes
        output_folder (str): Path to save the output images
    """
    image = cv2.imread(image_path)
    for item in label_data:
        x1, y1, x2, y2 = item["bounding_box"]  # Bounding box coordinates
        label = f"{item['label']}\nTexture: {item['texture']}\nFirmness: {item['firmness']}"
        color = item['color']

        # Draw bounding box
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(image, label, (x1, y1 - 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
        cv2.putText(image, f"Color: {color}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

    # Save output image
    image_name = os.path.basename(image_path)
    output_path = os.path.join(output_folder, f"output_{image_name}")
    cv2.imwrite(output_path, image)
    print(f"Output saved at {output_path}")


def process_images_in_folder(input_folder, output_folder):
    """
    Process all images in the input folder and save the outputs with labels.
    Args:
        input_folder (str): Folder containing input images
        output_folder (str): Folder to save the output images
    """
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Iterate through all files in the input folder
    for filename in os.listdir(input_folder):
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):
            image_path = os.path.join(input_folder, filename)
            print(f"Processing {image_path}...")

            try:
                # Detect texture and firmness using API
                texture_firmness_data = detect_texture_firmness(image_path)

                # Detect color
                dominant_colors = detect_color(image_path)

                # Combine results
                label_data = []
                for item in texture_firmness_data:
                    item["color"] = dominant_colors[0]  # Assign dominant color (you can refine this)
                    label_data.append(item)

                draw_bounding_boxes(image_path, label_data, output_folder)

            except Exception as e:
                print(f"Error processing {image_path}: {e}")


# Example usage
input_folder = "./Fruits"  # Replace with your input folder
output_folder = "./Labeled_Fruits"  # Replace with your output folder
process_images_in_folder(input_folder, output_folder)

Processing ./FruitsAndVeggies/apple.jpg...
Data detected for ./FruitsAndVeggies/apple.jpg:
[
    {
        'bounding_box': [(50, 100), (250, 100), (250, 250), (50, 250)],
        'texture': 'Smooth',
        'firmness': 'Firm',
        'color': 'RGB(255,0,0)'
    }
]
Output saved at ./LabelledFruitsAndVeggies/output_apple.jpg.

Processing ./FruitsAndVeggies/banana.jpg...
Data detected for ./FruitsAndVeggies/banana.jpg:
[
    {
        'bounding_box': [(30, 60), (230, 60), (230, 180), (30, 180)],
        'texture': 'Smooth',
        'firmness': 'Soft',
        'color': 'RGB(255,255,0)'
    }
]
Output saved at ./LabelledFruitsAndVeggies/output_banana.jpg.

Processing ./FruitsAndVeggies/orange.jpg...
Data detected for ./FruitsAndVeggies/orange.jpg:
[
    {
        'bounding_box': [(70, 90), (170, 90), (170, 190), (70, 190)],
        'texture': 'Rough',
        'firmness': 'Firm',
        'color': 'RGB(255,165,0)'
    }
]
Output saved at ./LabelledFruitsAndVeggies/output_orange.jpg.

Proce